# rfm_feature_engineering

In [1]:
import pandas as pd
import sys
sys.path.append('../src')
from rfm import create_rfm # module for create rfm table

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12


In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

## Calculating Recency, Frequency, Monetary

RFM Calculating code in rfm.py

analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1) # Date for reference for calculating recency

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,   # Recency
    'InvoiceNo': 'nunique',                                    # Frequency
    'Revenue': 'sum'                                           # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']


In [5]:
rfm = create_rfm(df)

In [6]:
# Recency score (lower recency = better → reverse scoring)
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

# Frequency score (higher frequency = better)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 
                         5, labels=[1,2,3,4,5])

# Monetary score (higher money = better)
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

In [7]:
rfm[['R_score','F_score','M_score']] = rfm[['R_score','F_score','M_score']].astype(int)


## Classifing customer base on RFM score

In [8]:
def segment_customer(row):
    
    # Best customers
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return 'Champions'
    
    # Good & consistent buyers
    elif row['F_score'] >= 3:
        return 'Loyal Customers'
    
    # Previously good but inactive recently
    elif row['R_score'] <= 2 and row['F_score'] >= 2:
        return 'At Risk'
    
    # Low activity customers
    else:
        return 'Lost'

In [9]:
rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,Segment
0,12346,326,1,77183.60,1,1,5,Lost
1,12347,2,7,4310.00,5,5,5,Champions
2,12348,75,4,1437.24,2,4,4,Loyal Customers
3,12349,19,1,1457.55,4,1,4,Lost
4,12350,310,1,294.40,1,1,2,Lost


In [10]:
rfm['Segment'].value_counts()

Segment
Loyal Customers    1464
Lost               1227
Champions          1136
At Risk             507
Name: count, dtype: int64

In [11]:
import os
path = os.makedirs("../data/processed", exist_ok=True) # Create results directory if it doesn't exist
rfm.to_csv("../data/processed/rfm_data.csv", index=False)

# END